BLEND OF CATBOOST AND LIGHTBGM

In [ ]:
%pip -q install catboost lightgbm

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.2 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
import lightgbm as lgb

DATA_DIR = '/content/drive/MyDrive/ML_project_dataset'
OUT = f'{DATA_DIR}/submission_blend_v2.csv'

for filename in ['train.csv', 'test.csv', 'submission.csv']:
    path = os.path.join(DATA_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing {path}. Check DATA_DIR and your Drive files.')

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
sub = pd.read_csv(f'{DATA_DIR}/submission.csv')
print('train:', train.shape, '| test:', test.shape, '| submission template:', sub.shape)

train: (18076, 12) | test: (4520, 11) | submission template: (4520, 2)


In [ ]:
def add_route(df):
    df = df.copy()
    df['route'] = df['source_city'].astype(str) + '_to_' + df['destination_city'].astype(str)
    return df

FEATS = [
    'airline', 'source_city', 'departure_time', 'stops', 'arrival_time',
    'destination_city', 'flight_class', 'duration', 'days_left', 'flight', 'route'
]

X = add_route(train)[FEATS].copy()
y = train['price'].copy()
X_test = add_route(test)[FEATS].copy()
CATS = [c for c in FEATS if not pd.api.types.is_numeric_dtype(X[c])]

print('Features:', FEATS)
print('Categorical features:', CATS)
print('X:', X.shape, '| X_test:', X_test.shape)

Features: ['airline', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'flight_class', 'duration', 'days_left', 'flight', 'route']
Categorical features: ['airline', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'flight_class', 'flight', 'route']
X: (18076, 11) | X_test: (4520, 11)


In [ ]:
# Model 1: CatBoost, depth 8
cb8 = CatBoostRegressor(
    iterations=800, depth=8, learning_rate=0.06,
    random_seed=42, verbose=0, cat_features=CATS
)
cb8.fit(X, y)

CatBoostRegressor(cat_features=['airline', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'flight_class', 'flight', 'route'], depth=8, iterations=800, learning_rate=0.06, loss_function='RMSE', random_seed=42, verbose=0)

In [ ]:
# Model 2: CatBoost, depth 6
cb6 = CatBoostRegressor(
    iterations=1000, depth=6, learning_rate=0.06,
    l2_leaf_reg=3, random_seed=42, verbose=0, cat_features=CATS
)
cb6.fit(X, y)

CatBoostRegressor(cat_features=['airline', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'flight_class', 'flight', 'route'], depth=6, iterations=1000, l2_leaf_reg=3, learning_rate=0.06, loss_function='RMSE', random_seed=42, verbose=0)

In [ ]:

# Model 3: LightGBM with native categorical columns
X_lgb = X.copy()
X_test_lgb = X_test.copy()
for col in CATS:
    category_type = pd.CategoricalDtype(sorted(set(X[col]) | set(X_test[col])))
    X_lgb[col] = X_lgb[col].astype(category_type)
    X_test_lgb[col] = X_test_lgb[col].astype(category_type)

lgb_model = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.03, num_leaves=31,
    min_child_samples=10, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, cat_smooth=20, verbose=-1, random_state=42
)
lgb_model.fit(X_lgb, y)
print('All three models trained on the full training dataset.')

All three models trained on the full training dataset.


In [ ]:
#Blending all three predictions
pred_cb8 = cb8.predict(X_test)
pred_cb6 = cb6.predict(X_test)
pred_lgb = lgb_model.predict(X_test_lgb)

# Blend weights from the supplied script: 40% CatBoost d8, 30% CatBoost d6, 30% LightGBM.
pred = 0.40 * pred_cb8 + 0.30 * pred_cb6 + 0.30 * pred_lgb
pred = np.clip(pred, 0, None)

submission = sub.copy()
submission['price'] = pred
assert submission.shape == (len(test), 2)
assert submission['id'].tolist() == test['id'].tolist(), 'ID order mismatch.'
assert submission['price'].notna().all(), 'Missing predictions found.'

print('Submission shape:', submission.shape)
display(submission.head())
display(submission['price'].describe())

Submission shape: (4520, 2)


,id,price
0,18395,78.039632
1,11771,179.928136
2,22326,355.172592
3,18332,120.452271
4,11021,535.062585


,price
count,4520.000000
mean,280.495965
std,188.636676
min,16.756051
25%,158.770845
50%,246.804555
75%,346.958733
max,2288.739956


In [ ]:
submission.to_csv(OUT, index=False)
print('Saved:', OUT)
print('Keep your existing rank-3 submission unchanged as a fallback; compare this submission before uploading.')

Saved: /content/drive/MyDrive/ML_project_dataset/submission_blend_v2.csv
Keep your existing rank-3 submission unchanged as a fallback; compare this submission before uploading.


**CB + LGBM - PART 2** (NO DIFFERENCE, JUST 5 FOLD VALIDATION ADDED )

In [ ]:


import os, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor
import lightgbm as lgb
warnings.filterwarnings("ignore")

DATA_DIR = "/content/drive/MyDrive/ML_project_dataset"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/mnt/user-data/uploads"
OUT_DIR = DATA_DIR if DATA_DIR.startswith("/content") else "/mnt/user-data/outputs"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/submission.csv")
print(train.shape, test.shape, sample_sub.shape)

(18076, 12) (4520, 11) (4520, 2)


In [ ]:
def add_route(d):
    d = d.copy()
    d["route"] = d["source_city"].astype(str) + "_to_" + d["destination_city"].astype(str)
    return d

FEATS = ["airline","source_city","departure_time","stops","arrival_time","destination_city",
         "flight_class","duration","days_left","flight","route"]

X  = add_route(train)[FEATS].reset_index(drop=True)
y  = train["price"].reset_index(drop=True)
Xt = add_route(test)[FEATS].reset_index(drop=True)
CATS = [c for c in FEATS if not pd.api.types.is_numeric_dtype(X[c])]
LEVELS = {c: sorted(set(X[c]) | set(Xt[c])) for c in CATS}   # category names only, no price information
print("categorical:", CATS)
print("train features:", X.shape, "| test features:", Xt.shape)

categorical: ['airline', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'flight_class', 'flight', 'route']
train features: (18076, 11) | test features: (4520, 11)


In [ ]:
def as_cat(df):
    df = df.copy()
    for c in CATS:
        df[c] = pd.Categorical(df[c], categories=LEVELS[c])
    return df

def make_model(name, seed=42):
    if name == "CatBoost d8":
        return CatBoostRegressor(iterations=800, depth=8, learning_rate=0.06, random_seed=seed,
                                 verbose=0, cat_features=CATS, thread_count=-1)
    if name == "CatBoost d6":
        return CatBoostRegressor(iterations=1000, depth=6, learning_rate=0.06, l2_leaf_reg=3,
                                 random_seed=seed, verbose=0, cat_features=CATS, thread_count=-1)
    if name == "LightGBM":
        return lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, num_leaves=31, min_child_samples=10,
                                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8, cat_smooth=20,
                                 verbose=-1, random_state=seed)

def fit_predict(name, X_tr, y_tr, X_te):
    m = make_model(name)
    if name.startswith("CatBoost"):
        m.fit(X_tr, y_tr); return m.predict(X_te)
    m.fit(as_cat(X_tr), y_tr); return m.predict(as_cat(X_te))

def metrics(y_true, p):
    return dict(RMSE=mean_squared_error(y_true, p)**.5, MAE=mean_absolute_error(y_true, p), R2=r2_score(y_true, p))

MODELS = ["CatBoost d8", "CatBoost d6", "LightGBM"]
WEIGHTS = {"CatBoost d8": 0.40, "CatBoost d6": 0.30, "LightGBM": 0.30}
def blend(preds, w=WEIGHTS):
    return sum(w[k] * preds[k] for k in w)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42)

rows, val_pred = [], {}
for name in MODELS:
    val_pred[name] = fit_predict(name, X_train, y_train, X_val)
    rows.append({"Model": name, **metrics(y_val, val_pred[name])})
rows.append({"Model": "Blend (40/30/30)", **metrics(y_val, blend(val_pred))})

val_table = pd.DataFrame(rows).set_index("Model").round(4)
val_table.loc["Random Forest (earlier notebook)"] = [144.739, 47.017, 0.6276]   # recorded result
val_table.loc["Poly degree 8 (original baseline)"] = [176.260, 86.435, 0.4477]   # recorded result
val_table

,RMSE,MAE,R2
Model,,,
CatBoost d8,140.7829,49.3047,0.6477
CatBoost d6,141.5311,50.7065,0.6439
LightGBM,144.1350,49.7454,0.6307
Blend (40/30/30),140.8390,48.5205,0.6474
Random Forest (earlier notebook),144.7390,47.0170,0.6276
Poly degree 8 (original baseline),176.2600,86.4350,0.4477


In [ ]:
#5-FOLD CROSS - VALIDATION

kf = KFold(n_splits=5, shuffle=True, random_state=42)
folds = list(kf.split(X))
oof = {}
for name in MODELS:
    t = time.time(); o = np.zeros(len(X))
    for tr_i, va_i in folds:
        o[va_i] = fit_predict(name, X.iloc[tr_i], y.iloc[tr_i], X.iloc[va_i])
    oof[name] = o
    print(f"{name:12s} done in {time.time()-t:.0f}s")

oof["Blend (40/30/30)"] = blend(oof)
cv_rows = []
for name, o in oof.items():
    fold_r2 = [r2_score(y.iloc[v], o[v]) for _, v in folds]
    cv_rows.append({"Model": name, **metrics(y, o), **{f"fold{i+1}": r for i, r in enumerate(fold_r2)}})
cv_table = pd.DataFrame(cv_rows).set_index("Model").round(4)
cv_table

CatBoost d8  done in 188s
CatBoost d6  done in 132s
LightGBM     done in 14s


,RMSE,MAE,R2,fold1,fold2,fold3,fold4,fold5
Model,,,,,,,,
CatBoost d8,122.8713,47.5475,0.6936,0.6452,0.7795,0.7782,0.7465,0.5897
CatBoost d6,123.3718,48.8076,0.6911,0.6457,0.7716,0.7781,0.7500,0.5828
LightGBM,125.8401,47.4719,0.6786,0.6333,0.7767,0.7504,0.7458,0.5637
Blend (40/30/30),121.7812,46.4170,0.6990,0.6481,0.7916,0.7809,0.7657,0.5861


In [1]:
import pandas as pd
from IPython.display import display

final_blend_results = [
    {"Model": "CatBoost depth 8", "Evaluation": "80/20 holdout", "RMSE": 140.7829, "MAE": 49.3047, "R2": 0.6477},
    {"Model": "CatBoost depth 6", "Evaluation": "80/20 holdout", "RMSE": 141.5311, "MAE": 50.7065, "R2": 0.6439},
    {"Model": "LightGBM", "Evaluation": "80/20 holdout", "RMSE": 144.1350, "MAE": 49.7454, "R2": 0.6307},
    {"Model": "Blend: 40% CatBoost d8 + 30% CatBoost d6 + 30% LightGBM", "Evaluation": "80/20 holdout", "RMSE": 140.8390, "MAE": 48.5205, "R2": 0.6474},
    {"Model": "Random Forest (reference from earlier notebook)", "Evaluation": "earlier holdout; reference", "RMSE": 144.7390, "MAE": 47.0170, "R2": 0.6276},
    {"Model": "Polynomial degree 8 (reference)", "Evaluation": "earlier holdout; reference", "RMSE": 176.2600, "MAE": 86.4350, "R2": 0.4477},
    {"Model": "CatBoost depth 8", "Evaluation": "5-fold OOF", "RMSE": 122.8713, "MAE": 47.5475, "R2": 0.6936},
    {"Model": "CatBoost depth 6", "Evaluation": "5-fold OOF", "RMSE": 123.3718, "MAE": 48.8076, "R2": 0.6911},
    {"Model": "LightGBM", "Evaluation": "5-fold OOF", "RMSE": 125.8401, "MAE": 47.4719, "R2": 0.6786},
    {"Model": "Blend: 40/30/30", "Evaluation": "5-fold OOF", "RMSE": 121.7812, "MAE": 46.4170, "R2": 0.6990},
]
display(pd.DataFrame(final_blend_results).sort_values(["Evaluation", "RMSE"]).round(4))

,Model,Evaluation,RMSE,MAE,R2
9,Blend: 40/30/30,5-fold OOF,121.7812,46.4170,0.6990
6,CatBoost depth 8,5-fold OOF,122.8713,47.5475,0.6936
7,CatBoost depth 6,5-fold OOF,123.3718,48.8076,0.6911
8,LightGBM,5-fold OOF,125.8401,47.4719,0.6786
0,CatBoost depth 8,80/20 holdout,140.7829,49.3047,0.6477
3,Blend: 40% CatBoost d8 + 30% CatBoost d6 + 30%...,80/20 holdout,140.8390,48.5205,0.6474
1,CatBoost depth 6,80/20 holdout,141.5311,50.7065,0.6439
2,LightGBM,80/20 holdout,144.1350,49.7454,0.6307
4,Random Forest (reference from earlier notebook),earlier holdout; reference,144.7390,47.0170,0.6276
5,Polynomial degree 8 (reference),earlier holdout; reference,176.2600,86.4350,0.4477
